# MERRA-2 Trapping Variables

The original weather dataset has four variables — temperature, humidity, and the two wind
components. None of them measure the thing that actually traps pollution over the LA basin.

Los Angeles sits in a bowl under a persistent subsidence inversion. Warm air aloft caps the
cooler marine air below, and everything emitted underneath that cap stays there. Temperature
and wind are *correlated* with this, which is why the models work at all, but they never
observe it directly.

Four new variables, all subsets of MERRA-2 products already used by this project:

| variable | product | what it gives us |
|---|---|---|
| `PBLH` | M2T1NXFLX | planetary boundary layer height — the depth of air pollution can mix into |
| `T850` | M2T1NXSLV | temperature at 850 hPa; against surface temperature this gives inversion strength |
| `SLP` | M2T1NXSLV | sea-level pressure — high pressure means subsidence and stagnation |
| `TQV` | M2T1NXSLV | total column water vapor |

From these come the two features that matter most physically:

- **Inversion strength** = `T850 − T2M`. Positive means warmer air sits above cooler air — a
  temperature inversion, the cap.
- **Ventilation index** = `PBLH × wind speed`. The standard operational air-quality metric: how
  much air volume is available per unit time to dilute emissions. Low ventilation is the
  classic smog setup.

Output: `data/processed/la_daily_trapping_2016_2025.csv`

## Setup

In [1]:
import glob, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

FLX       = ROOT / 'data' / 'raw' / 'merra2_flx'
SLVX      = ROOT / 'data' / 'raw' / 'merra2_slv_extra'
PROCESSED = ROOT / 'data' / 'processed'
OUT_CSV   = PROCESSED / 'la_daily_trapping_2016_2025.csv'

for d, what in [(FLX, 'PBLH granules'), (SLVX, 'T850/SLP/TQV granules')]:
    assert d.exists(), (f'{what} missing at {d}. These are gitignored (~150 MB) — '
                        'see the README for the download command.')

flx_files  = sorted(glob.glob(str(FLX  / '*.nc4')))
slvx_files = sorted(glob.glob(str(SLVX / '*.nc4')))
print(f'PBLH granules:        {len(flx_files)}')
print(f'T850/SLP/TQV granules: {len(slvx_files)}')

PBLH granules:        3652
T850/SLP/TQV granules: 3652


## Aggregating hourly to daily

Same convention as the main weather pipeline: spatial mean over the 3×3 grid to get one value
per hour, then daily statistics over the 24 hours.

`PBLH` gets special treatment. Its daily **minimum** matters more than its mean — the shallowest
the mixing layer gets is what determines how concentrated pollution becomes overnight and in the
early morning. The night-time mean is tracked separately for the same reason.

In [2]:
def summarize_flx(path):
    with xr.open_dataset(path) as ds:
        pblh = ds.PBLH.mean(dim=('lat', 'lon')).values
        date = pd.Timestamp(ds.time.values[0]).normalize()
        hours = pd.DatetimeIndex(ds.time.values).hour
    night = pblh[(hours < 7) | (hours >= 20)]
    return {'date': date,
            'pblh_mean':  pblh.mean(),
            'pblh_min':   pblh.min(),
            'pblh_max':   pblh.max(),
            'pblh_night': night.mean()}


def summarize_slvx(path):
    with xr.open_dataset(path) as ds:
        t850 = ds.T850.mean(dim=('lat', 'lon')).values - 273.15
        slp  = ds.SLP.mean(dim=('lat', 'lon')).values / 100.0    # Pa -> hPa
        tqv  = ds.TQV.mean(dim=('lat', 'lon')).values
        date = pd.Timestamp(ds.time.values[0]).normalize()
    return {'date': date,
            't850_mean': t850.mean(), 't850_max': t850.max(),
            'slp_mean': slp.mean(), 'slp_max': slp.max(),
            'tqv_mean': tqv.mean()}


def run(files, fn, label):
    rows, failures = [], []
    t0 = time.time()
    for i, f in enumerate(files, 1):
        try:
            rows.append(fn(f))
        except Exception as e:
            failures.append((Path(f).name, repr(e)))
        if i % 750 == 0 or i == len(files):
            el = time.time() - t0
            print(f'  {label} {i:5}/{len(files)}  {el:5.1f}s  ~{el/i*(len(files)-i):4.1f}s left  '
                  f'({len(failures)} failures)')
    if failures:
        print(f'  {len(failures)} failures, first few:')
        for name, err in failures[:5]:
            print('   ', name, '->', err)
    return pd.DataFrame(rows)


flx_df  = run(flx_files,  summarize_flx,  'PBLH')
slvx_df = run(slvx_files, summarize_slvx, 'SLVX')
print(f'\nPBLH rows {len(flx_df)}, SLVX rows {len(slvx_df)}')

  PBLH   750/3652    2.9s  ~11.3s left  (0 failures)


  PBLH  1500/3652    5.6s  ~ 8.1s left  (0 failures)


  PBLH  2250/3652    8.4s  ~ 5.2s left  (0 failures)


  PBLH  3000/3652   11.1s  ~ 2.4s left  (0 failures)


  PBLH  3652/3652   13.4s  ~ 0.0s left  (0 failures)


  SLVX   750/3652    3.6s  ~13.9s left  (0 failures)


  SLVX  1500/3652    7.3s  ~10.4s left  (0 failures)


  SLVX  2250/3652   10.9s  ~ 6.8s left  (0 failures)


  SLVX  3000/3652   14.6s  ~ 3.2s left  (0 failures)


  SLVX  3652/3652   17.7s  ~ 0.0s left  (6 failures)
  6 failures, first few:
    slvx_20240512.nc4 -> ValueError("did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:\nhttps://docs.xarray.dev/en/stable/getting-started-guide/installing.html\nhttps://docs.xarray.dev/en/stable/user-guide/io.html")
    slvx_20240515.nc4 -> ValueError("did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:\nhttps://docs.xarray.dev/en/stable/getting-started-guide/installing.html\nhttps://docs.xarray.dev/en/stable/user-guide/io.html")
    slvx_20240517.nc4 -> ValueError("did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy'].

## Joining to the existing weather series

The derived features need surface temperature and wind speed from the original pipeline, so
everything is merged on date. An inner join keeps only days where all three sources agree,
which is the honest thing to do — a partially-populated row would silently poison the lags
downstream.

In [3]:
weather = pd.read_csv(PROCESSED / 'la_daily_weather_2016_2025.csv', parse_dates=['date'])

trap = (weather[['date', 't2m_mean', 't2m_max', 'wind_speed_mean']]
        .merge(flx_df,  on='date', how='inner')
        .merge(slvx_df, on='date', how='inner'))

print(f'weather {len(weather)} | PBLH {len(flx_df)} | SLVX {len(slvx_df)} -> joined {len(trap)}')
print(f'range: {trap.date.min().date()} -> {trap.date.max().date()}')

full_range = pd.date_range(trap.date.min(), trap.date.max(), freq='D')
gaps = full_range.difference(trap.date)
print(f'gaps inside the joined range: {len(gaps)}')
if len(gaps):
    print('  first few:', [str(d.date()) for d in gaps[:8]])

weather 3652 | PBLH 3652 | SLVX 3646 -> joined 3646
range: 2016-01-01 -> 2025-12-30
gaps inside the joined range: 6
  first few: ['2024-05-12', '2024-05-15', '2024-05-17', '2024-05-18', '2024-05-20', '2024-05-21']


## The derived physics

Two features carry most of the intended signal:

- **`inversion_strength` = T850 − T2M.** Positive means air at 850 hPa (roughly 1.5 km up) is
  *warmer* than the surface — a capping inversion. The stronger it is, the more firmly pollution
  is held down.
- **`ventilation_index` = PBLH × wind speed.** Mixing depth times transport speed. This is the
  operational metric forecasters actually use; low values are the smog setup.

`pblh_range` is added as a proxy for how vigorously the boundary layer breathes over the day.

In [4]:
trap['inversion_strength']     = trap['t850_mean'] - trap['t2m_mean']
trap['inversion_strength_max'] = trap['t850_max']  - trap['t2m_max']
trap['ventilation_index']      = trap['pblh_mean'] * trap['wind_speed_mean']
trap['ventilation_index_min']  = trap['pblh_min']  * trap['wind_speed_mean']
trap['pblh_range']             = trap['pblh_max']  - trap['pblh_min']

NEW_COLS = ['pblh_mean', 'pblh_min', 'pblh_max', 'pblh_night', 'pblh_range',
            't850_mean', 'slp_mean', 'tqv_mean',
            'inversion_strength', 'inversion_strength_max',
            'ventilation_index', 'ventilation_index_min']

print(trap[NEW_COLS].describe().T.round(2).to_string())
print(f'\ndays with a capping inversion (T850 > T2M): '
      f'{(trap.inversion_strength > 0).mean():.1%}')

                         count     mean      std     min      25%      50%      75%      max
pblh_mean               3646.0   630.96   221.85  124.57   490.94   613.85   747.90  2031.79
pblh_min                3646.0   192.58   155.06   62.09    73.80   141.80   251.40  1510.14
pblh_max                3646.0  1420.47   327.01  303.40  1215.16  1393.29  1604.96  3354.95
pblh_night              3646.0   765.83   244.04  153.98   618.01   753.07   895.58  2197.50
pblh_range              3646.0  1227.89   309.91  217.54  1018.38  1208.19  1406.30  3135.61
t850_mean               3646.0    15.01     7.41   -2.34     9.05    14.89    21.28    33.27
slp_mean                3646.0  1013.78     4.02  998.57  1010.81  1013.16  1016.61  1030.53
tqv_mean                3646.0    14.87     7.03    2.02    10.02    13.40    18.20    49.99
inversion_strength      3646.0    -2.24     2.90   -9.09    -4.42    -1.99    -0.15     6.56
inversion_strength_max  3646.0    -6.10     2.44  -11.32    -7.98    -

## Do these actually track AQI?

The whole justification for this work is that these variables measure the trapping mechanism
directly. If they do, they should correlate with AQI at least as strongly as the temperature
features already in the model.

In [5]:
aqi = pd.read_csv(PROCESSED / 'la_daily_aqi_5pollutants_v2_2016_2025.csv',
                  parse_dates=['date'])[['date', 'daily_aqi', 'dominant_pollutant']]
chk = trap.merge(aqi, on='date', how='inner')

corr = chk[NEW_COLS + ['daily_aqi']].corr()['daily_aqi'].drop('daily_aqi')
print('Correlation with daily AQI (new variables):')
print(corr.sort_values(key=abs, ascending=False).round(3).to_string())

print('\nFor comparison, the strongest existing feature:')
base = pd.read_csv(PROCESSED / 'la_modeling_dataset.csv', parse_dates=['date'])
print(f"  t2m_max  {base[['t2m_max','daily_aqi']].corr().iloc[0,1]:.3f}")

print('\nSplit by which pollutant set the daily max:')
for p in ['Ozone', 'PM2.5']:
    sub = chk[chk.dominant_pollutant == p]
    c = sub[NEW_COLS + ['daily_aqi']].corr()['daily_aqi'].drop('daily_aqi')
    top = c.sort_values(key=abs, ascending=False).head(4)
    print(f'  {p} (n={len(sub)}): ' + ', '.join(f'{k} {v:+.3f}' for k, v in top.items()))

Correlation with daily AQI (new variables):
t850_mean                 0.753
inversion_strength        0.670
inversion_strength_max    0.487
slp_mean                 -0.373
ventilation_index        -0.267
ventilation_index_min    -0.242
tqv_mean                  0.223
pblh_night                0.169
pblh_range                0.133
pblh_max                  0.116
pblh_mean                 0.116
pblh_min                 -0.021

For comparison, the strongest existing feature:
  t2m_max  0.705

Split by which pollutant set the daily max:
  Ozone (n=1659): t850_mean +0.756, inversion_strength +0.611, inversion_strength_max +0.444, slp_mean -0.409
  PM2.5 (n=1853): inversion_strength +0.527, t850_mean +0.449, ventilation_index -0.375, pblh_mean -0.330


## Saving

In [6]:
# t850_max is carried in the CSV but is NOT a model feature. It is needed to recompute
# inversion_strength_max exactly downstream (models/predict.py); deriving it back from the
# rounded CSV values instead would introduce ~1e-4 drift.
HELPER_COLS = ['t850_max']
out = trap[['date'] + NEW_COLS + HELPER_COLS].copy()
out.to_csv(OUT_CSV, index=False)
print(f'Saved {OUT_CSV.relative_to(ROOT)}  ({len(out)} rows, {out.shape[1]} columns)')
out.head()

Saved data/processed/la_daily_trapping_2016_2025.csv  (3646 rows, 14 columns)


,date,pblh_mean,pblh_min,pblh_max,pblh_night,pblh_range,t850_mean,slp_mean,tqv_mean,inversion_strength,inversion_strength_max,ventilation_index,ventilation_index_min,t850_max
0,2016-01-01,683.150696,369.128815,1290.024902,805.143005,920.896118,2.620809,1019.532227,5.992043,-5.181899,-10.062286,2955.958630,1597.201777,4.024475
1,2016-01-02,398.559662,148.717194,1026.890137,529.213196,878.172974,6.589863,1018.202454,8.467620,-2.989520,-7.918640,1194.994340,445.896114,7.965637
2,2016-01-03,462.951996,174.536285,1162.332520,563.672485,987.796265,6.826126,1015.718750,13.018421,-3.307156,-7.055817,1533.809406,578.257354,7.755005
3,2016-01-04,639.365295,324.507782,1278.396484,687.432190,953.888672,5.291673,1009.528809,19.362478,-5.864977,-8.659424,2456.670741,1246.875267,6.196655
4,2016-01-05,692.375488,239.892548,1113.083374,911.780273,873.190796,5.214868,1010.402649,20.033081,-6.042310,-6.693176,3072.725236,1064.630244,6.391052


## What this told us

**`t850_mean` correlates with daily AQI at 0.753 — stronger than any feature in the original
dataset**, including `t2m_max` at 0.705. Temperature 1.5 km above the surface predicts LA's air
quality better than temperature at the surface does.

That is not as strange as it first sounds. 850 hPa temperature is a clean measure of the air mass
itself, whereas surface temperature is contaminated by the marine layer, local wind and time of day.
When a high-pressure ridge parks over Southern California, the air aloft warms by subsidence — and
that same subsidence is what caps the basin.

`inversion_strength` (T850 − T2M) follows at 0.670, and it behaves exactly as the physics predicts:

| dominant pollutant | strongest new features |
|---|---|
| Ozone (n=1,659) | `t850_mean` +0.756, `inversion_strength` +0.611 |
| PM2.5 (n=1,853) | **`inversion_strength` +0.527**, `t850_mean` +0.449, `ventilation_index` −0.375 |

For ozone, upper-air warmth dominates — it is a photochemical, heat-driven pollutant. For PM2.5 the
ordering flips and **inversion strength becomes the top correlate**, with ventilation index close
behind. PM2.5 is emitted rather than manufactured, so what matters is whether the basin is capped and
still. Two different pollutants, two different mechanisms, visible directly in the correlations.

**Raw `PBLH` is disappointing on its own** (0.116 for the mean, −0.021 for the daily minimum), which
is worth flagging because boundary-layer height is usually the first variable anyone reaches for. It
only becomes useful once combined with wind into `ventilation_index` (−0.267 overall, −0.375 on PM2.5
days). Mixing depth alone does not tell you much; mixing depth times transport speed does.

A capping inversion is present on 23.3% of days — roughly one day in four, which matches the known
climatology of the LA basin and is a good sign the calculation is right.